In [1]:
import os
import sys
import json
import time
import random
import pathlib
from openai import OpenAI

In [2]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [3]:
BATCH_ID_FILE  = "batch_id.txt"
REQUESTS_FILE  = "batch_requests.jsonl"
OUTPUT_FILE    = "data/tool_calling.jsonl"
MODEL          = "gpt-4o-mini"
MAX_TOKENS     = 4000

In [4]:
SYSTEM_PROMPT = (
    "আপনি e-porcha.com এর একজন সহায়ক AI assistant। "
    "প্রশ্ন বাংলায় হলে বাংলায় উত্তর দিন, ইংরেজিতে হলে ইংরেজিতে উত্তর দিন। "
    "প্রয়োজনে tools ব্যবহার করুন।"
)

In [5]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_khatian",
            "description": (
                "Search for land khatian (খতিয়ান) records by district, upazila, "
                "mouza, and khatian number. Returns owner name, land area, and plot details."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "district": {"type": "string", "description": "District name in Bengali or English e.g. ঢাকা, Dhaka"},
                    "upazila":  {"type": "string", "description": "Upazila name"},
                    "mouza":    {"type": "string", "description": "Mouza name"},
                    "khatian_number": {"type": "string", "description": "Khatian number (খতিয়ান নম্বর)"},
                },
                "required": ["district", "khatian_number"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_plot",
            "description": (
                "Search for land plot (দাগ) information by district, upazila, mouza "
                "and plot number (দাগ নম্বর). Returns ownership, area, and land type."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "district":    {"type": "string", "description": "District name"},
                    "upazila":     {"type": "string", "description": "Upazila name"},
                    "mouza":       {"type": "string", "description": "Mouza name"},
                    "plot_number": {"type": "string", "description": "Plot/dag number (দাগ নম্বর)"},
                },
                "required": ["district", "plot_number"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "verify_namjari",
            "description": (
                "Verify e-Namjari (ই-নামজারি) application status by application number. "
                "Returns current status, applicant name, and processing details."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "application_number": {"type": "string", "description": "Namjari application number (আবেদন নম্বর)"},
                    "district":           {"type": "string", "description": "District name (optional)"},
                },
                "required": ["application_number"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "download_mouza_map",
            "description": (
                "Get the download link or availability status of a Mouza map (মৌজা ম্যাপ) "
                "for a specific area."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "district": {"type": "string", "description": "District name"},
                    "upazila":  {"type": "string", "description": "Upazila name"},
                    "mouza":    {"type": "string", "description": "Mouza name"},
                },
                "required": ["district", "mouza"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_faq",
            "description": (
                "Search the e-porcha FAQ and knowledge base for answers to general questions "
                "about land records, processes, fees, and procedures."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The user's question or topic to search for"},
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_service_fees",
            "description": (
                "Get the official fee structure for e-porcha services such as khatian copy, "
                "namjari application, map download, etc."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "service_type": {
                        "type": "string",
                        "description": "Type of service e.g. khatian_copy, namjari, mouza_map, porcha_copy",
                        "enum": ["khatian_copy", "namjari", "mouza_map", "porcha_copy", "mutation"]
                    },
                },
                "required": ["service_type"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_land_owner",
            "description": (
                "Check the current owner of a land plot by NID number or owner name "
                "along with district and plot/khatian info."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "nid_number": {"type": "string", "description": "National ID number of the owner"},
                    "owner_name": {"type": "string", "description": "Name of the owner in Bengali"},
                    "district":   {"type": "string", "description": "District name"},
                    "khatian_number": {"type": "string", "description": "Khatian number if known"},
                },
                "required": ["district"]
            }
        }
    },
]

In [6]:
TOOL_TOPICS = [
    # (topic_description, tool_hint, sample_count)
    ("খতিয়ান অনুসন্ধান — district/upazila/mouza/khatian number দিয়ে",         "search_khatian",  180),
    ("দাগ নম্বর দিয়ে জমির তথ্য খোঁজা",                                         "search_plot",     150),
    ("নামজারি আবেদনের স্ট্যাটাস যাচাই",                                         "verify_namjari",  120),
    ("মৌজা ম্যাপ ডাউনলোড করার অনুরোধ",                                          "download_mouza_map", 100),
    ("e-porcha সম্পর্কে সাধারণ প্রশ্ন যেগুলো FAQ থেকে উত্তর দেওয়া যায়",       "search_faq",      150),
    ("সেবার ফি জানতে চাওয়া — খতিয়ান কপি, নামজারি, মৌজা ম্যাপ",               "get_service_fees", 100),
    ("জমির মালিকানা NID বা নাম দিয়ে যাচাই",                                    "check_land_owner", 100),
    ("multi-turn — প্রথমে জিজ্ঞেস করে, তারপর তথ্য দিয়ে tool call হয়",          "any",              100),
]

In [7]:
NO_TOOL_TOPICS = [
    "e-porcha কী এবং এটি কীভাবে কাজ করে তা জানতে চাওয়া",
    "জমির রেজিস্ট্রেশন এবং মিউটেশনের পার্থক্য",
    "খতিয়ান এবং দাগ নম্বরের মধ্যে পার্থক্য কী",
    "নামজারি করতে কী কী কাগজপত্র লাগে",
    "ই-পর্চা ওয়েবসাইটে কীভাবে একাউন্ট খুলতে হয়",
    "পর্চা কী জিনিস",
    "জমির CS, SA, RS, BS খতিয়ান কী",
    "মৌজা ম্যাপ কী কাজে লাগে",
    "ভূমি উন্নয়ন কর কীভাবে দিতে হয়",
    "জমির দলিল হারিয়ে গেলে কী করতে হয়",
]

In [8]:
def fake_tool_result(tool_name, arguments):
    if tool_name == "search_khatian":
        return {
            "khatian_number": arguments.get("khatian_number", "১২৩"),
            "owner_name": "মোঃ আব্দুল করিম",
            "district": arguments.get("district", "ঢাকা"),
            "land_area": "০.৩৫ একর",
            "land_type": "কৃষি জমি",
            "plot_numbers": ["৪৫৬", "৪৫৭"],
            "status": "found"
        }
    elif tool_name == "search_plot":
        return {
            "plot_number": arguments.get("plot_number", "৪৫৬"),
            "owner_name": "মোঃ রহিম উদ্দিন",
            "land_area": "০.১২ একর",
            "land_type": "বাস্তু জমি",
            "khatian_number": "২১৫",
            "status": "found"
        }
    elif tool_name == "verify_namjari":
        return {
            "application_number": arguments.get("application_number", "NM-2024-12345"),
            "status": "প্রক্রিয়াধীন",
            "applicant_name": "মোসাম্মৎ রহিমা বেগম",
            "submitted_date": "২০২৪-০৩-১৫",
            "current_step": "সহকারী কমিশনার (ভূমি) পর্যায়ে"
        }
    elif tool_name == "download_mouza_map":
        return {
            "mouza": arguments.get("mouza", ""),
            "district": arguments.get("district", ""),
            "available": True,
            "download_url": "https://eporcha.gov.bd/map/download/12345",
            "format": "PDF",
            "size": "2.3 MB"
        }
    elif tool_name == "search_faq":
        return {
            "query": arguments.get("query", ""),
            "answer": "e-porcha হলো বাংলাদেশ সরকারের ভূমি মন্ত্রণালয়ের অনলাইন সেবা পোর্টাল যেখানে জমির খতিয়ান, পর্চা, মৌজা ম্যাপ এবং নামজারি সংক্রান্ত সেবা পাওয়া যায়।",
            "source": "e-porcha FAQ",
            "confidence": 0.92
        }
    elif tool_name == "get_service_fees":
        fees = {
            "khatian_copy": "১০০ টাকা প্রতি কপি",
            "namjari": "১,০০০ টাকা আবেদন ফি",
            "mouza_map": "২০০ টাকা প্রতি শিট",
            "porcha_copy": "১০০ টাকা প্রতি কপি",
            "mutation": "১,০০০ টাকা"
        }
        service = arguments.get("service_type", "khatian_copy")
        return {"service": service, "fee": fees.get(service, "তথ্য পাওয়া যায়নি")}
    elif tool_name == "check_land_owner":
        return {
            "owner_name": "মোঃ জহিরুল ইসলাম",
            "nid": arguments.get("nid_number", ""),
            "district": arguments.get("district", ""),
            "khatian_numbers": ["৩৪৫", "৩৪৬"],
            "total_land": "০.৭৮ একর",
            "status": "found"
        }
    return {"error": "Unknown tool"}

In [17]:
def build_tool_prompt(topic, tool_hint, is_multiturn=False):
    tools_summary = "\n".join(
        f"- {t['function']['name']}: {t['function']['description'][:80]}"
        for t in TOOLS
    )
    multiturn_note = (
        "\n\nএটি একটি multi-turn conversation হবে। "
        "প্রথম বার্তায় user প্রয়োজনীয় তথ্য না দিলে assistant জিজ্ঞেস করবে, "
        "তারপর user তথ্য দিলে tool call হবে।"
        if is_multiturn else ""
    )
    return f"""আপনি e-porcha.com chatbot এর জন্য training data তৈরি করছেন।

e-porcha হলো বাংলাদেশের অনলাইন ভূমি সেবা পোর্টাল।

Available tools:
{tools_summary}

বিষয়: {topic}{multiturn_note}

নিচের নিয়ম মেনে ৫টি আলাদা conversation তৈরি করুন:

১. User এর প্রশ্ন স্বাভাবিক বাংলায় হবে — বাস্তব Bangladeshi ভূমি সমস্যা
২. Tool call এর arguments সঠিক এবং realistic হবে
৩. Tool result পাওয়ার পর assistant এর উত্তর সংক্ষিপ্ত ও প্রাসঙ্গিক হবে
৪. Suggested tool: {tool_hint} (যদি প্রযোজ্য হয়)
৫. বিভিন্ন জেলার নাম ব্যবহার করুন: ঢাকা, চট্টগ্রাম, সিলেট, রাজশাহী, খুলনা, বরিশাল, ময়মনসিংহ, রংপুর

শুধুমাত্র JSON array return করুন, অন্য কিছু নয়:

[
  {{
    "messages": [
      {{"role": "user", "content": "..."}},
      {{"role": "assistant", "content": null, "tool_calls": [{{"id": "call_001", "type": "function", "function": {{"name": "tool_name", "arguments": "{{...}}"}}}}]}},
      {{"role": "tool", "tool_call_id": "call_001", "content": "{{...}}"}},
      {{"role": "assistant", "content": "বাংলায় সংক্ষিপ্ত উত্তর"}}
    ]
  }}
]"""


def build_no_tool_prompt(topic):
    return f"""আপনি e-porcha.com chatbot এর জন্য training data তৈরি করছেন।

বিষয়: {topic}

এই ধরনের প্রশ্নের জন্য কোনো tool call দরকার নেই — সরাসরি উত্তর দিতে হবে।

৫টি conversation তৈরি করুন যেখানে:
১. User সরাসরি প্রশ্ন করে
২. Assistant tool call না করে সরাসরি বাংলায় উত্তর দেয়
৩. উত্তর সংক্ষিপ্ত ও তথ্যপূর্ণ হবে

শুধুমাত্র JSON array return করুন:

[
  {{
    "messages": [
      {{"role": "user", "content": "..."}},
      {{"role": "assistant", "content": "সরাসরি বাংলায় উত্তর"}}
    ]
  }}
]"""

In [9]:
def build_requests():
    """Build all batch request lines and return as list of dicts."""
    requests = []
    request_meta = []  # tracks (custom_id → topic, is_tool, is_multiturn)

    # Tool calling requests
    for topic, tool_hint, count in TOOL_TOPICS:
        is_multiturn = (tool_hint == "any")
        n_batches = -(-count // 5)  # ceiling division

        for i in range(n_batches):
            custom_id = f"tool_{len(requests):05d}"
            prompt = build_tool_prompt(topic, tool_hint, is_multiturn)
            requests.append({
                "custom_id": custom_id,
                "method":    "POST",
                "url":       "/v1/chat/completions",
                "body": {
                    "model":       MODEL,
                    "max_tokens":  MAX_TOKENS,
                    "temperature": 0.9,
                    "messages": [
                        {"role": "user", "content": prompt}
                    ],
                },
            })
            request_meta.append({"custom_id": custom_id, "is_tool": True, "topic": topic})

    return requests, request_meta

In [10]:
def cmd_submit():
    print("Building batch requests...")
    requests, meta = build_requests()
    print(f"Total requests to submit: {len(requests)}")

    # Save meta for collect step
    with open("batch_meta.jsonl", "w", encoding="utf-8") as f:
        for m in meta:
            f.write(json.dumps(m, ensure_ascii=False) + "\n")

    # Write requests JSONL
    with open(REQUESTS_FILE, "w", encoding="utf-8") as f:
        for req in requests:
            f.write(json.dumps(req, ensure_ascii=False) + "\n")

    print(f"Uploading {REQUESTS_FILE} to OpenAI Files API...")
    with open(REQUESTS_FILE, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")
    print(f"File uploaded: {uploaded.id}")

    print("Submitting batch job...")
    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"project": "eporcha-tool-data"},
    )
    print(f"Batch submitted!")
    print(f"  batch_id : {batch.id}")
    print(f"  status   : {batch.status}")

    with open(BATCH_ID_FILE, "w") as f:
        f.write(batch.id)
    print(f"batch_id saved to {BATCH_ID_FILE}")

In [11]:
def _load_batch_id():
    if not pathlib.Path(BATCH_ID_FILE).exists():
        print(f"ERROR: {BATCH_ID_FILE} not found. Run `submit` first.")
        sys.exit(1)
    return pathlib.Path(BATCH_ID_FILE).read_text().strip()

def cmd_status():
    batch_id = _load_batch_id()
    batch = client.batches.retrieve(batch_id)
    counts = batch.request_counts
    print(f"Batch ID  : {batch.id}")
    print(f"Status    : {batch.status}")
    print(f"Requests  : total={counts.total}  completed={counts.completed}  failed={counts.failed}")
    if batch.output_file_id:
        print(f"Output file: {batch.output_file_id}")
    if batch.error_file_id:
        print(f"Error file : {batch.error_file_id}  ← download manually if needed")
    return batch.status

In [15]:
def is_valid_sample(sample, expect_tool_call=True):
    messages = sample.get("messages", [])

    if len(messages) < 2:
        return False

    # Must end with assistant
    if messages[-1].get("role") != "assistant":
        return False

    # Assistant final response must have content
    if not messages[-1].get("content", "").strip():
        return False

    if expect_tool_call:
        # Must have at least one tool call turn
        has_tool_call = any(
            msg.get("role") == "assistant" and msg.get("tool_calls")
            for msg in messages
        )
        if not has_tool_call:
            return False

        # tool_call_id must match
        call_ids = set()
        for msg in messages:
            if msg.get("role") == "assistant" and msg.get("tool_calls"):
                for tc in msg["tool_calls"]:
                    call_ids.add(tc.get("id"))
        for msg in messages:
            if msg.get("role") == "tool":
                if msg.get("tool_call_id") not in call_ids:
                    return False

        # arguments must be valid JSON string
        for msg in messages:
            if msg.get("role") == "assistant" and msg.get("tool_calls"):
                for tc in msg["tool_calls"]:
                    try:
                        json.loads(tc["function"]["arguments"])
                    except Exception:
                        return False

    # Check has Bangla characters
    full_text = " ".join(
        msg.get("content") or "" for msg in messages
    )
    has_bangla = any('\u0980' <= c <= '\u09FF' for c in full_text)
    if not has_bangla:
        return False

    return True

def finalize_sample(sample, include_tools=True):
    messages = sample["messages"]

    # Prepend system message
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + messages

    result = {"messages": messages}
    if include_tools:
        result["tools"] = TOOLS

    return result

def parse_response_body(body, is_tool):
    """Extract and validate samples from a single completion response body."""
    try:
        text = body["choices"][0]["message"]["content"].strip()
    except (KeyError, IndexError, TypeError):
        return []

    # Strip ```json fences`
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1])

    try:
        raw_samples = json.loads(text)
    except json.JSONDecodeError:
        return []

    valid = []
    for s in raw_samples:
        if is_valid_sample(s, expect_tool_call=is_tool):
            valid.append(finalize_sample(s, include_tools=True))
    return valid


In [12]:
def cmd_collect():
    batch_id = _load_batch_id()
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        print(f"Batch not yet completed. Current status: {batch.status}")
        print("Run `python generate_tool_data_batch.py status` to check again.")
        # sys.exit(1)

    if not batch.output_file_id:
        print("No output file found. Check error_file_id if available.")
        # sys.exit(1)

    # Load meta
    meta_map = {}
    if pathlib.Path("batch_meta.jsonl").exists():
        with open("batch_meta.jsonl", encoding="utf-8") as f:
            for line in f:
                m = json.loads(line)
                meta_map[m["custom_id"]] = m

    print(f"Downloading results from file {batch.output_file_id}...")
    content = client.files.content(batch.output_file_id)
    lines = content.text.strip().split("\n")
    print(f"Downloaded {len(lines)} result lines.")

    all_samples = []
    errors = 0

    for line in lines:
        if not line.strip():
            continue
        try:
            result = json.loads(line)
        except json.JSONDecodeError:
            errors += 1
            continue

        custom_id = result.get("custom_id", "")
        meta = meta_map.get(custom_id, {})
        is_tool = meta.get("is_tool", True)

        body = result.get("response", {}).get("body", {})
        if result.get("error"):
            errors += 1
            continue

        samples = parse_response_body(body, is_tool)
        all_samples.extend(samples)

    random.shuffle(all_samples)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for s in all_samples:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")

    tool_count   = sum(1 for s in all_samples if any(
        msg.get("tool_calls") for msg in s["messages"] if msg.get("role") == "assistant"
    ))
    notool_count = len(all_samples) - tool_count

    print("=" * 50)
    print(f"Total valid samples : {len(all_samples)}")
    print(f"  Tool-calling      : {tool_count}")
    print(f"  No-tool negatives : {notool_count}")
    print(f"  Parse errors      : {errors}")
    print(f"Saved to            : {OUTPUT_FILE}")
    print("=" * 50)

In [18]:
cmd_submit()

Building batch requests...
Total requests to submit: 200
Uploading batch_requests.jsonl to OpenAI Files API...
File uploaded: file-UzEifuK6JioPQ72HduzSK5
Submitting batch job...
Batch submitted!
  batch_id : batch_69d1fb12c42c81909aed14c08e931b7a
  status   : validating
batch_id saved to batch_id.txt


In [19]:
cmd_status()

Batch ID  : batch_69d1fb12c42c81909aed14c08e931b7a
Status    : completed
Requests  : total=200  completed=200  failed=0
Output file: file-UqwfzM9CD1PSG2muJE5Vjr


'completed'

In [20]:
cmd_collect()

Downloaded 200 result lines.
Total valid samples : 570
  Tool-calling      : 570
  No-tool negatives : 0
  Parse errors      : 0
Saved to            : tool_calling.jsonl
